In [ ]:
# From 6_s_wave_side_Jj_Peierls_AsIn
def plot_bands_lead(lead, name, L, s, xlim, ylim, nk, p):
    with Timer(name):
        bands = kwant.physics.Bands(lead, params=p)
        k_range = np.linspace(-xlim, xlim, nk)
        E = np.array([bands(k) for k in k_range])
        plt.figure(figsize=(s,s))
        for i in range(E.shape[1]):
            plt.plot(k_range, E[:, i]*1e3, 'k.', markersize=1)
        info = (f"s-wave \n{name} \nL = {L} nm \n$\mu = {p['mu']*1e3}$ meV \n$B = {p['B']}$ T\
        \n$\Delta = {p['delta']*1e3}$ meV")
        plt.text(1.05*xlim, 0, info, fontsize=3*s)
        plt.axhline(p['delta']*1e3, color='blue', label='$\Delta$', lw=0.2)
        plt.axhline(p['mu']*1e3, color='green', label='$\mu$', lw=0.3)
        plt.axhline(-p['delta']*1e3, color='blue', lw=0.2)
        plt.xlabel("$k_y$ [1/a]"); plt.ylabel("$E [meV]$")
        plt.xlim(-xlim, xlim); plt.ylim(-ylim, ylim)
        plt.yticks(np.arange(-ylim, ylim+ylim/10, ylim/5))
        plt.xticks(np.arange(-xlim, xlim+xlim/10, xlim/2))
        plt.legend(loc='upper right', fontsize=2*s); plt.grid(alpha=0.3); plt.show()

In [ ]:
%%px --local

@lru_cache(maxsize=1)
def initialize_lead(_a, _W, _W_sc_block, _L, _t):
    lat = kwant.lattice.square(_a, norbs=2) 
    lead_hybrid = kwant.Builder(kwant.TranslationalSymmetry((0, -_a)), particle_hole=tau_y)
    lead_hybrid[lat.shape(lambda pos: -_W <= pos[0] < 0, (-_a, 0))] = onsite
    if _W_sc_block > 0: lead_hybrid[lat.shape(lambda pos: 0 <= pos[0] < _W_sc_block, (0, 0))] = onsite_sc
    lead_hybrid[lat.neighbors()] = hop
    return lead_hybrid.finalized()
#-------------------------------------------------------------------------------------------------------------------
def get_modes_lead_sys(sys, delta, x0, mu, B, N, E):
    with Timer('modes_lead'):
        lead = sys.leads[0]
        mu_vec = np.linspace(mu[0], mu[1], N)
        B_vec = np.linspace(B[0], B[1], N)
        points = [(m, b) for m in mu_vec for b in B_vec]
        results = lview.map_sync(lambda p: compute_modes_lead_sys(lead, delta, x0, p[0], p[1], E), points)
        map_v = np.array(results).reshape(N, N)
    return map_v
#-------------------------------------------------------------------------------------------------------------------
def compute_modes_lead_sys(lead, _delta, _x0, _mu, _b, _E):
    p = dict(mu=_mu, B=_b, delta=_delta, x0=_x0)
    prop_modes, _ = lead.modes(energy=_E, params=p)
    return len(prop_modes.momenta)
#-------------------------------------------------------------------------------------------------------------------
def compute_eigen_energies(_k, _sys_wrap, _modes, _mu, _B, _delta, _x0):
    p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0, k_y=_k)
    H = _sys_wrap.hamiltonian_submatrix(params=p, sparse=True)
    evals = eigsh(H, k=_modes, sigma=0, which='LM', return_eigenvectors=False)
    return np.sort(evals.real)
#-------------------------------------------------------------------------------------------------------------------
def get_modes_wrapped(_a, _W, _W_sc_wrap, _L, _t, _delta, _x0, _mu, _B, N, nk, modes, xlim, tol):
    with Timer('modes_wrapped'):
        k_range = np.linspace(0, xlim, nk)
        mu_vec = np.linspace(_mu[0], _mu[1], N)
        B_vec = np.linspace(_B[0], _B[1], N)
        map_v = np.zeros((N, N), dtype=int)

        for i in range(N):
            for j in range(N):
                count = 0
                mu_i = mu_vec[i]
                B_i = B_vec[j]
                spectrum_at_k = lview.map_sync(lambda k: compute_eigen_energies(k, initialize(_a, _W, _W_sc_wrap, _L, _t),
                                                                        modes, mu_i, B_i, _delta, _x0), k_range)
                spectrum_at_k = np.array(spectrum_at_k)
                for idx in range(spectrum_at_k.shape[1]):
                    if np.any(abs(spectrum_at_k[:, idx]) < tol): count += 2
                map_v[i, j] = count
    return map_v
#-------------------------------------------------------------------------------------------------------------------
map_wrapped = get_modes_wrapped(_a=a, _W=W1000, _W_sc_wrap=Wscwrap1000, _L=L400, _t=t, _delta=0.001, _x0=0.0, _mu=(0, 0.01), _B=(0, 1.5), 
                                N=50, nk=101, modes=10, xlim=PI, tol=0.2*1e-4)
#-------------------------------------------------------------------------------------------------------------------
print(f"Punkt i, j: map_lead \t map_wrapped")
for i in range(50):
            for j in range(50):
                print(f"Punkt {i}, {j}: \t{map_lead[i, j]} \t {map_wrapped[i, j]}")
#-------------------------------------------------------------------------------------------------------------------
def get_dk_wrap(E, k_range):
    k_l, k_p = -PI, PI
    prev = PI
    n_k = len(k_range)
    n_modes = E.shape[1]
    start = n_k//2 if n_k%2 == 0 else (n_k+1)//2 # we do not want the k=0 point since it is not possible unless delta==0?
    for i in range(start, n_k):
        curr_k = k_range[i]
        for m in range(n_modes-1):
            E1, E2 = E[i, m], E[i, m+1]
            if E1*E2 < 0 and abs(E2) < prev:
                prev = E2
                k_p = curr_k
    dk = 2 * k_p; k_l = -k_p
    print(f"delta_k = {dk:.5f} \t")
    return dk
#-------------------------------------------------------------------------------------------------------------------
def show_E_k(Evals, k_range):
    n_k = len(k_range)
    n_modes = Evals.shape[1]

    for i in range(n_k - 1):
        if abs(k_range[i]) < 0.4:
            print(f"\nk_{i} =  {k_range[i]}")
            for m in range(10):
                print(f"E_{i}_{m} = {Evals[i, m]*1e3}\t")
#-------------------------------------------------------------------------------------------------------------------
dk_wrap = get_dk_wrap(E=Evals_400, k_range=k_400)

In [ ]:
def plot_mu_conductance(sys, mu, barrier):
    g_normal = []
    g_andreev = []
    
    # Stałe parametry - liczymy dla E=0
    params_norm = dict(delta=0)
    params_andreev = dict(delta=0.001)
    
    for i in mu:
        params_norm['mu'] = i
        params_andreev['mu'] = i
        g_normal.append(compute_conductance(0, sys, params_norm))
        g_andreev.append(compute_conductance(0, sys, params_andreev))

    plt.figure(figsize=(3, 3))
    plt.plot(mu, g_normal, label='Normal $\Delta=0, G=1*N$', color='blue')
    plt.plot(mu, g_andreev, label='Andreev $\Delta>0, G=2*N$', color='red')
    plt.xlabel("$\mu$ [eV]")
    plt.ylabel("G [$e^2/h$]")
    plt.legend(fontsize=6)
    plt.grid(True); 
    plt.show()
#-------------------------------------------------------------------------------------------------------------------
def plot_mu_conductance(sys, mu, p):
    
    g = []
    for i in mu:
        p['mu'] = i
        g.append(compute_Gj(sys, E=0, p=p, j=0))


    plt.figure(figsize=(3, 3))
    plt.plot(mu, g)
    plt.xlabel("$\mu$ [eV]")
    plt.ylabel("G [$e^2/h$]")
    plt.grid(True); plt.show()

plot_mu_conductance(sys=h_1000, mu = np.linspace(0, 0.01, 100), p=params_edge)

In [ ]:
%%px --local

def compute_LDOC(sys, E, p):
    with Timer('compute_LDOC'):
        modes = sys.leads[0].modes(energy=E, params=p)[0]
        channels = len(modes.momenta) // 2
        print(f"For energy = {E:.3f} | {channels} open channels")
        res = lview.map_sync(lambda n: kwant.operator.Density(sys, s_z)(kwant.wave_function(sys, E, params=p)(0)[n]), range(channels))
        return sum(res)

In [ ]:
def plot_analytical_G(_a, _W, _W_sc_wrap, _L, _t, _delta, _x0, mu_range, B_val,
                      N, _alpha, _beta, _modes, _xlim, _nk):
    k_range = np.linspace(-_xlim, _xlim, _nk)
    mu_vec = np.linspace(mu_range[0], mu_range[1], N)
    mu_dk = [(x, y) for x in mu_vec for y in k_range]

    def eigen_mu(args):
        mu_val, k_val = args
        wrap = initialize_wrapped(_a, _W, _W_sc_wrap, _L, _t)
        E, _ = compute_eigen(k_val, wrap, _modes, mu_val, B_val, _delta, _x0)
        return E
    def analytical(alpha, beta, L, a, dk): 
        return 1.0 - 8.0*(alpha*beta)**2 * np.sin(dk*L/a/2.0)**2
    results = lview.map_sync(eigen_mu, mu_dk)
    E_mu_dk = np.array(results).reshape(N, _nk, _modes)
    dk_vec = np.zeros(N)
    for i in range(N):
        dk_vec[i] = get_dk(E_mu_dk[i], k_range)
        
    g = lview.map_sync(lambda dks: analytical(alpha=_alpha, beta=_beta, L=_L, a=_a, dk=dks), dk_vec)
    return np.array(g)
#-------------------------------------------------------------------------------------------------------------------
def plot_Gj(sys, E, name, L, s, p, j):
    with Timer('plot_Gj'):
        with Timer('compute_Gj'):
            G = lview.map_sync(lambda e: compute_Gj(sys, e, p, j), E)
        plt.figure(figsize=(s, s))
        plt.plot(E*1e3, G)
        info = (f"s-wave \n{name} \nL = {L} nm \n$\mu = {p['mu']*1e3:.2}$ meV \n$B = {p['B']:.2}$ T\
        \n$\Delta = {p['delta']*1e3:.2}$ meV")
        plt.text(1.05, 0.5, info, transform=plt.gca().transAxes, fontsize=3*s, verticalalignment='center')
        plt.axvline(p['delta']*1e3, color='blue', label='$\Delta$', lw=0.2)
        plt.axvline(p['mu']*1e3, color='green', label='$\mu$', lw=0.3)
        plt.axvline(-p['delta']*1e3, color='blue', lw=0.2)
        plt.xlabel("$E$ [meV]"); plt.ylabel("$G$ [$e^2/h$]")
        plt.legend(loc='upper right', fontsize=2*s); plt.grid(); plt.show()
#-------------------------------------------------------------------------------------------------------------------
def plot_LDOC(sys, E, name, L, s, p):
    with Timer('plot_LDOC'):
        psi2 = compute_LDOC(sys, E, p=p)
        kwant.plotter.map(sys, psi2, cmap='seismic', show=False, fig_size=(s, s/2), vmin=-max(abs(psi2)), vmax=max(abs(psi2)))
        info = (f"s-wave \n{name} L = {L} nm $|\Psi|^2$ ($E = {E*1e3:.3}$ meV) $\mu = {p['mu']*1e3:.2}$ meV $B = {p['B']:.2}$ T\
        $\Delta = {p['delta']*1e3:.2}$ meV")
        plt.text(-0.2, 1.5, info, transform=plt.gca().transAxes, fontsize=1.5*s)
        plt.xlabel("x [nm]"); plt.ylabel("y [nm]")
        plt.show()

In [ ]:
import matplotlib.colors as mcolors
def plot_modes_map(map_v, mu_range, B_range, name):
    unique_modes = np.unique(map_v)
    n_colors = len(unique_modes)
    base_cmap = plt.cm.get_cmap('viridis', n_colors)
    custom_cmap = mcolors.ListedColormap(base_cmap(np.linspace(0, 1, n_colors)))
    plt.figure(figsize=(10, 8))
    im = plt.imshow(map_v.T, origin='lower', aspect='auto', 
                    extent=[mu_range[0]*1e3, mu_range[1]*1e3, B_range[0], B_range[1]],
                    cmap=custom_cmap)
    cbar = plt.colorbar(im, ticks=unique_modes)
    cbar.set_label('Liczba dostępnych modów $N$')
    plt.title(f"Mapa modów: {name}")
    plt.xlabel(r'$\mu$ [meV]')
    plt.ylabel(r'$B$ [T]')
    plt.grid(alpha=0.1); plt.show()

plot_modes_map(V_map, (0, 0.01), (0, 1.5), "Metoda LEAD")

In [ ]:
#-------------------------------------------------------------------------------------------------------------------
def interactive_G(_sys, _name, _L, _s, _j, nE, Elim, _mu, _B, _delta, _x0):
    Evals = np.linspace(-Elim, Elim, nE)
    _p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0)
    plot_Gj(sys=_sys, E=Evals, name=_name, L=_L, s=_s, p=_p, j=_j)
#-------------------------------------------------------------------------------------------------------------------
def interactive_LDOC(_sys, Eval, _name, _L, _s, _mu, _B, _delta, _x0):
    _p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0)
    plot_LDOC(sys=_sys, E=Eval, name=_name, L=_L, s=_s, p=_p)

In [ ]:
LDOS_HYBRID = interactive(
    interactive_LDOC,
    _sys=fixed(hybrid_1000),
    Eval=widgets.FloatSlider(min=0, max=0.002, step=0.0001, value=0.0002, description='Eval:', readout_format='.4f'),
    _name=fixed('HYBRID_1000'),
    _L=fixed(L),
    _s=fixed(10),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.0001, value=0.002, description='mu:', readout_format='.4f'),
    _B=widgets.FloatSlider(min=-5, max=0, step=0.05, value=-0.3, description='B:', readout_format='.2f'),
    _delta=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.001, description='delta:', readout_format='.3f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.0f')
);
controls = VBox(LDOS_HYBRID.children[:-1])
output = LDOS_HYBRID.children[-1]
display(HBox([output, controls]))

G_HYBRID= interactive(
    interactive_G,
    _sys=fixed(hybrid_1000),
    _name=fixed('HYBRID_1000'),
    _L=fixed(L),
    _s=fixed(4),
    _j=fixed(0),
    nE=fixed(100),
    Elim=widgets.FloatSlider(min=0, max=0.05, step=0.001, value=0.002, description='E:', readout_format='.4f'),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.0001, value=0.002, description='mu:', readout_format='.4f'),
    _B=widgets.FloatSlider(min=-5, max=0, step=0.05, value=-0.3, description='B:', readout_format='.2f'),
    _delta=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.001, description='delta:', readout_format='.3f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.0f')
);
controls = VBox(G_HYBRID.children[:-1])
output = G_HYBRID.children[-1]
display(HBox([output, controls]))

In [ ]:
G_BLOCK = interactive(
    interactive_G,
    _sys=fixed(block_1000),
    _name=fixed('BLOCK_1000'),
    _L=fixed(L),
    _s=fixed(4),
    _j=fixed(0),
    nE=fixed(100),
    Elim=widgets.FloatSlider(min=0, max=0.05, step=0.001, value=0.002, description='E:', readout_format='.4f'),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.0001, value=0.002, description='mu:', readout_format='.4f'),
    _B=widgets.FloatSlider(min=-5, max=0, step=0.05, value=-0.3, description='B:', readout_format='.2f'),
    _delta=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.001, description='delta:', readout_format='.3f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.0f')
);
-------------------------------------------------------------------------------------------------------------------
controls = VBox(G_BLOCK.children[:-1])
output = G_BLOCK.children[-1]
display(HBox([output, controls]))

LDOS_BLOCK = interactive(
    interactive_LDOC,
    _sys=fixed(block_1000),
    Eval=widgets.FloatSlider(min=0, max=0.002, step=0.0001, value=0.0002, description='Eval:', readout_format='.4f'),
    _name=fixed('BLOCK_1000'),
    _L=fixed(L),
    _s=fixed(10),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.0001, value=0.002, description='mu:', readout_format='.4f'),
    _B=widgets.FloatSlider(min=-5, max=0, step=0.05, value=-0.3, description='B:', readout_format='.2f'),
    _delta=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.001, description='delta:', readout_format='.3f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.0f')
);
controls = VBox(LDOS_BLOCK.children[:-1])
output = LDOS_BLOCK.children[-1]
display(HBox([output, controls]))